In [ ]:
# !pip install langchain==0.2.7 langchain-core==0.2.23 langchain-community==0.2.5 langchain-openai==0.1.8 chromadb==0.5.0 sentence-transformers langchain_groq 

In [ ]:
# Python = 3.10
# langchain = 0.2.7
# langchain-core = 0.2.23
# langchain-community = 0.2.5
# langchain-openai = 0.1.8
# chromadb = 0.5.0
# protobuf = 4.25.3
# sentence-transformers

In [ ]:
# brew install ollama
# ollama serve
# ollama pull llama3.1
# ollama run llama3.1

In [24]:
import warnings
warnings.filterwarnings('ignore')

In [1]:
from langchain_community.llms import Ollama
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain.chains import RetrievalQA

In [2]:
import os

for var in ["HTTP_PROXY", "HTTPS_PROXY", "http_proxy", "https_proxy"]:
    if var in os.environ:
        del os.environ[var]

os.environ["NO_PROXY"] = "*"

In [3]:
filepath = "./data/vector-db/data.txt"

In [6]:
loader = TextLoader(filepath)

In [7]:
docs = loader.load()

In [8]:
docs

[Document(metadata={'source': './data/vector-db/data.txt'}, page_content='Deep learning is a subset of machine learning that uses neural networks.\nTransformers are the backbone of modern LLMs.\nVector databases store embeddings for semantic search.')]

In [10]:
splitter = RecursiveCharacterTextSplitter(
                        chunk_size=50,
                        chunk_overlap=30
                )

In [11]:
chunks = splitter.split_documents(docs)

In [12]:
chunks

[Document(metadata={'source': './data/vector-db/data.txt'}, page_content='Deep learning is a subset of machine learning that'),
 Document(metadata={'source': './data/vector-db/data.txt'}, page_content='of machine learning that uses neural networks.'),
 Document(metadata={'source': './data/vector-db/data.txt'}, page_content='Transformers are the backbone of modern LLMs.'),
 Document(metadata={'source': './data/vector-db/data.txt'}, page_content='Vector databases store embeddings for semantic'),
 Document(metadata={'source': './data/vector-db/data.txt'}, page_content='store embeddings for semantic search.')]

In [18]:
# splitter1 = RecursiveCharacterTextSplitter(
#                         chunk_size=30,
#                         chunk_overlap=30
#                 )
# chunks1 = splitter1.split_documents(docs)
# chunks1

In [20]:
embedding = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

/opt/anaconda3/envs/RAG_1/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:139: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  warn_deprecated(


In [23]:
vector_db = Chroma.from_documents(
                    documents = chunks,
                    embedding = embedding,
                    collection_name = "rag-text"
                )

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [26]:
retriever = vector_db.as_retriever(search_kwargs={'k': 3})

In [36]:
llm = Ollama(
    model="llama3.1",
    temperature=2
)

In [30]:
# RetrievalQA.from_chain_type??

In [37]:
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    verbose=True
)

In [38]:
query = "What does the document say about vector databases?"

In [34]:
response_t_1_2 = qa.run(query)



> Entering new RetrievalQA chain...

> Finished chain.


In [35]:
response_t_1_2

'Vector databases store embeddings for semantic representations.'

In [39]:
response_t_2_0 = qa.run(query)
response_t_2_0



> Entering new RetrievalQA chain...

> Finished chain.


'According to the context provided:\n\nThe document mentions that "vector databases store embeddings for semantic of machine learning that uses neural networks." This suggests that the purpose of a vector database, as mentioned in this snippet, is to store and manage embedding data related to machine learning concepts involving neural networks. The main focus seems to be on their association with neural networks and their applications in handling semantic meanings.\n\nAs there isn\'t explicit information given about storage capacities, types of embeddings, query capabilities, or anything that typically differentiates databases across the board, it can be inferred that this snippet focuses on a niche application or use case where vector data is paramount—most likely related to applications like natural language processing or similar AI disciplines. \n\nThus, without more context about other aspects (e.g., types of operations supported beyond storage), what this document primarily says a